# 🎯 RQ1: Attribution Stability Under Prompt Perturbations
### Robustness Evaluation of 5 XAI Methods Against Surface, Syntactic, and Semantic Perturbations

---
**Research Question 1:** To what extent do token attribution rankings generated by XAI methods remain stable under meaning-preserving input text perturbations?

Evaluated Methods:
- **Gradient $\times$ Input** (analytical first-order gradient)
- **Occlusion** (Leave-One-Out)
- **Integrated Gradients** ($m = 25$ Riemann steps)
- **LIME** ($N = 150$ neighborhood samples)
- **SHAP** ($B = 1500$ permutation evaluations)


## 1. Environment Configuration and GPU Verification


In [ ]:
!nvidia-smi

import os
IN_COLAB = "google.colab" in str(get_ipython())
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    RESULTS_DIR = "/content/drive/MyDrive/llm_xai_results/rq1_stability"
else:
    RESULTS_DIR = "results/rq1_stability"

os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Results directory: {RESULTS_DIR}")


## 2. Execute RQ1 Stability Benchmark (Auto-Resume & Checkpointing)


In [ ]:
#@title RQ1 Experiment Parameters
samples_rq1 = 100 #@param {type:"integer"}
max_run_samples = 50 #@param {type:"integer"}
model_name = "microsoft/phi-2" #@param ["microsoft/phi-2", "meta-llama/Llama-3.1-8B", "Qwen/Qwen2.5-7B"]
methods = "GradientXInput,Occlusion,IntegratedGradients,LIME,SHAP" #@param {type:"string"}

!python scripts/run_experiments.py \
    --samples {samples_rq1} \
    --max-run-samples {max_run_samples} \
    --model {model_name} \
    --methods {methods} \
    --output-dir {RESULTS_DIR}


## 3. Statistical Analysis of RQ1 Results


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

raw_path = os.path.join(RESULTS_DIR, "rq1_stability_raw.csv")
if not os.path.exists(raw_path):
    raw_path = "results/rq1_stability/rq1_stability_raw.csv"

df = pd.read_csv(raw_path)
df_valid = df[df["is_preserved"] == True].copy()
print(f"Loaded {len(df_valid)} decision-preserved sentence pairs.")

# Summary metrics table
summary = df_valid.groupby("explainer_name").agg(
    spearman_mean=("spearman", "mean"),
    spearman_sem=("spearman", "sem"),
    kendall_mean=("kendall", "mean"),
    jaccard_k3_mean=("jaccard_k3", "mean"),
    orig_time_mean=("orig_time_s", "mean")
).round(3)
display(summary)


## 4. Stability Distribution Plots (Boxplots)


In [ ]:
plt.figure(figsize=(10, 5), dpi=300)
palette = {"GradientXInput": "#e41a1c", "Occlusion": "#377eb8", "IntegratedGradients": "#4daf4a", "LIME": "#984ea3", "SHAP": "#ff7f00"}

sns.boxplot(data=df_valid, x="explainer_name", y="spearman", palette=palette, boxprops=dict(alpha=0.8))
plt.title("RQ1: Attribution Ranking Stability (Spearman $\\rho$)", fontsize=12, fontweight="bold")
plt.xlabel("XAI Method", fontsize=11)
plt.ylabel(r"Spearman Rank Correlation $\\rho$", fontsize=11)
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "rq1_spearman_boxplot.png"))
plt.show()
